In [ ]:
VISION_MODEL = "gpt-4.1-mini"
FRAMES_PER_SECOND = 1  # Extract 1 frame per second
SCENE_CHANGE_THRESHOLD = 0.3  # Normalized 0-1

In [ ]:
from typing import Dict, Any

class MockOpenAIClient:
    def __init__(self):
        self.call_count = 0

    def chat_completions_create(self, model, messages, **kwargs):
        self.call_count += 1
        return {
            "choices": [{"message": {"content": "Frame analysis: This frame shows..."}}]
        }

_mock_client = MockOpenAIClient()

def get_openai_client():
    return _mock_client

def reset_mock_client():
    global _mock_client
    _mock_client = MockOpenAIClient()


In [ ]:
import cv2
from typing import List, Dict
from config import FRAMES_PER_SECOND, SCENE_CHANGE_THRESHOLD


class FrameExtractor:
    """Extracts frames from video files."""

    def __init__(self):
        """Initialize the FrameExtractor."""
        pass

    def extract_frames(self, video_path: str) -> List[Dict]:
        """
        Extract frames with metadata.
        """
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)

        frame_interval = int(fps / FRAMES_PER_SECOND)

        frames = []
        frame_num = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_num % int(frame_interval) == 0:
                timestamp = frame_num / fps
                frames.append({
                    "frame": frame,
                    "timestamp": timestamp,
                    "frame_number": frame_num
                })

            frame_num += 1

        cap.release()
        return frames


In [ ]:
import time
from typing import List, Dict
from api_client import get_openai_client
from config import VISION_MODEL


class FrameAnalyzer:
    """Analyzes video frames using Vision API."""

    def __init__(self):
        """Initialize the FrameAnalyzer."""
        self.client = get_openai_client()

    def analyze_frames(self, frames: List[Dict]) -> List[Dict]:
        """
        Analyze frames with Vision API.
        """
        results = []
        for frame_data in frames:

            timestamp = frame_data.get("timestamp", 0)
            prompt = f"""
                Analyze this frame at timestamp {timestamp:.2f} seconds.
                Describe what you see.
            """

            try:
                time.sleep(0.1)  # Rate limit example

                response = self.client.chat_completions_create(
                    model=VISION_MODEL,
                    messages=[{"role": "user", "content": prompt}]
                )

                results.append({
                    "analysis": response["choices"][0]["message"]["content"],
                    "timestamp": timestamp
                })

            except Exception as e:
                results.append({"analysis": f"Error: {str(e)}", "timestamp": timestamp})
                continue

        return results


In [ ]:
from typing import List, Dict, Optional


class VideoSummarizer:
    """Generates video summaries with chapter markers."""

    def __init__(self):
        """Initialize the VideoSummarizer."""
        pass

    def generate_summary(self, frame_analyses: List[Dict], audio_transcript: Optional[str] = None) -> Dict:
        """
        Generate summary with chapters.
        """
        sorted_analyses = sorted(frame_analyses, key=lambda x: x.get("timestamp", 0))

        combined_context = "\n".join([a["analysis"] for a in sorted_analyses])
        if audio_transcript:
            combined_context += f"\n\nAudio transcript: {audio_transcript}"

        chapters = []
        for i, analysis in enumerate(sorted_analyses):
            timestamp = analysis.get("timestamp", 0)
            minutes = int(timestamp // 60)
            seconds = int(timestamp % 60)
            chapters.append({
                "title": f"Chapter {i+1}",
                "timestamp": f"{minutes:02d}:{seconds:02d}"
            })

        return {
            "summary": f"Video analysis: {combined_context[:200]}...",
            "chapters": chapters
        }
